# YANA FPGA accelerator tutorial (KR260)
This notebook is a step-by-step guide to using the YANA SNN FPGA accelerator on the KR260 platform. It shows how to load the bitstream, initialize the driver, and run inference on a sample dataset. Follow the instructions in each cell to explore the accelerator.

**Hint:** Before using this notebook, your training & deployment artifacts need to be transferred to the FPGA board.

### Initialize the Accelerator
The following cell initializes the FPGA accelerator. It uses the `pynq` library to talk to the programmable logic (PL) and loads the bitstream. The `yana_fpga` module provides a high-level Python interface for configuration, data transfer, and inference. This notebook focuses on that API rather than low-level register programming.

In [ ]:
# Ignore me, this disables line wrapping in Jupyter Lab output cells
from IPython.display import display, HTML
display(HTML("<style>div.jp-OutputArea-output pre {white-space: pre;}</style>"))

# This is where the real tutorial begins
from pynq import PL
import yana_fpga

PL.reset()

accel = yana_fpga.YanaFpga(bitstream="design_1_wrapper.xsa") # Rename xsa file string if needed

The `YanaFpga` instance `accel` is the high-level interface to the SNN accelerator: it loads the bitstream, programs on-chip memories, streams input events, and reads outputs.

### The Safety Button

༼ つ ◕_◕ ༽つ*Don't execute this cell if you're simply following the tutorial.*

Whenever you want to reset the notebook kernel for any reason, execute the following cell first.




In [3]:
# Safety Button
#  Use before restarting the notebook kernel
PL.reset()
del accel

### Interacting with the Accelerator

In previous tutorial stages, you trained SNNs that meet the accelerator's hardware constraints. A subsequent deployment step converted your trained network and a few test samples into simple text-based file formats.

These files are read by `yana_fpga` when loading networks and running experiments. They contain the following information:

- **Network Representation:** Describes how the SNN is laid out within the accelerator's on-chip memory (e.g., weights, routing).
- **Input Samples:** Contains spike data for a couple of input samples taken from the dataset's test set.
- **Expected Output:** For each input sample, this file provides the raw output expected from the accelerator. This reference output was generated previously using a Python simulator that emulates the accelerator's computations and was then transferred to the FPGA board from your machine.

Your network and dataset files should reside within the directory `experiments`. The contents within the directory have to follow the following structure:
```
experiments
└── <dataset_name>
    ├── networks
    │   ├── <network_name>
    │   │   ├── <memory_files>.txt
    │   │   └── output_traces
    │   │       └── <expected_output_traces>.txt
    └── test_samples
        ├── <testset_samples>.txt
        └── sample_info.yaml
```

Beyond your own personal network and dataset files, we provide an example network based on NMNIST within the directory `experiments/example`. We will use this network to learn how to interact with the accelerator:

Loading the memory files is as simple as executing accel.load_network().

In [ ]:
# Load all memory files for the neural network
dataset_name = "example"
network_name = "network_nmnist"

_ = accel.load_network(dataset_name, network_name)

Before evaluating test samples, set the control registers as needed. See `yana_fpga.py` (control register map near the top of the class) for the YANA accelerator CSR layout.


In [3]:
accel.ctrl_reset = False # The accelerator is initialized in reset, take it out of reset
accel.num_output_neurons = accel._get_num_output_neurons(dataset_name, network_name)
accel.ctrl_enable = True

The accelerator is now ready to process one sample at a time from a trace file (continuous streaming input is not covered here).

Using `accel.evaluate_sample_trace_file()`, you can process a single sample and evaluate its results:

In [ ]:
# There are 10 smaples (0 to 8) in the example files.
# Feel free to evaluate the results of the other samples, too.
# Simply change the following index and run this cell again.
sample_index = 5

# Ignore this code for now. ( ͡° ͜ʖ ͡°)
samples = accel.experiment_tracker.parse_samples_yaml(
    f"experiments/{dataset_name}/test_samples/sample_info.yaml"
)

# Evaluate the sample
try:
    accel.evaluate_sample_trace_file(
        file_path=f"experiments/{dataset_name}/test_samples/sample_{sample_index}.txt",
        sample_index=sample_index,
        dataset_name=dataset_name,
        network_name=network_name
    )
except Exception as e:
    print(f"Error evaluating sample trace file: {e}")

The results displayed above include:
*   Processing statistics for the evaluated sample.
*   Output values that determine the final classification.
*   Comparison against the Python simulator's expected outputs using Mean Squared Error (MSE).
*   The number of clock cycles the accelerator took to process the sample.

By comparing the results of different samples, you can observe a key characteristic of event-based processing: the number of processing cycles is proportional to the number of input events in the sample.

### Batched Processing and Experiment Tracking

`YanaFpga` includes an `ExperimentTracker` (in `experiment_tracker.py`) to collect and summarize performance statistics across multiple samples for a given network and dataset combination (an "experiment"). This tracker records metrics like:

*   Total input events processed
*   Total clock cycles consumed
*   Classification accuracy
*   Mean Squared Error (MSE) compared to expected outputs

The `accel.run_experiment()` method processes a batch of test samples defined in a `sample_info.yaml` file (parsed using `experiment_tracker.parse_samples_yaml`), updates the tracker with the results from each sample, and finally, `accel.print_experiment_results()` displays a summary table.

Let's see how this works, first using the example network from before, and then for the networks trained by yours truly.


In [ ]:
# Reset the experiment tracker
accel.reset_experiment_tracker()

# Run experiments on the example dataset
accel.run_experiment("example", "network_nmnist", verbose=False)

# Print the results table
accel.print_experiment_results()

The final results table summarizes the experiment, showing the following metrics aggregated across all processed samples:
*   Total processed input events
*   Total timesteps
*   Total clock cycles used for processing
*   Overall network classification accuracy
*   Mean Squared Error (MSE) compared to the expected outputs from the Python simulator.

### Running Your Personal Experiments

We have now covered the essential technical details for using the accelerator via this notebook. The next code cell will automatically evaluate all the experiments (network and dataset combinations) that you previously transferred onto the FPGA board.

As you examine the results, pay attention to the impact of network pruning. You should observe that applying pruning to SNNs reduces the number of event-based computations performed by the accelerator. Consequently, pruned networks typically require less processing time compared to their corresponding base networks when evaluated on the same test dataset.

This marks the end of the guided part of this tutorial. Thank you for attending!

If you have time left, explore the notebook and the `yana_fpga` API further. Feel free to ask questions about the accelerator or the concepts covered.


In [ ]:
import os

# Reset the experiment tracker if needed
accel.reset_experiment_tracker()

# Iterate through the experiments folder
for dataset_name in os.listdir("experiments"):
    if dataset_name == "example": # Skip the 'example' dataset
        continue
    dataset_path = os.path.join("experiments", dataset_name)
    if os.path.isdir(dataset_path):
        networks_path = os.path.join(dataset_path, "networks")
        if os.path.isdir(networks_path):
            for network_name in os.listdir(networks_path):
                network_path = os.path.join(networks_path, network_name)
                if os.path.isdir(network_path):
                    # Run experiment for each network
                    try:
                        print(f"Running experiment: {dataset_name}/{network_name}")
                        accel.run_experiment(dataset_name, network_name, verbose=False)
                    except Exception as e:
                        print(f"Error running experiment for {dataset_name}/{network_name}: {e}")

# Print experiment results
accel.print_experiment_results()
